In [2]:
# =========================
# Standard library imports
# =========================
import sys
import os
from pathlib import Path
import math
import logging
import random
import subprocess
import warnings
import collections

from IPython.display import display

warnings.filterwarnings("ignore")


# =========================
# Numeric / stats
# =========================
import numpy as np
import pandas as pd
from scipy import stats
from scipy import __version__ as scipy_version


# =========================
# Plotting / visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns


# =========================
# scikit-learn: split / CV / metrics
# =========================
from sklearn import model_selection, metrics
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    KFold,
    GroupKFold,
    cross_validate,
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)


# =========================
# scikit-learn: models
# =========================
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn import __version__ as sklearn_version


# =========================
# Optimization
# =========================
import optuna


# =========================
# LightGBM
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e


# =========================
# XGBoost
# =========================
try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except Exception as e:
    raise RuntimeError(
        "XGBoost not installed. Please install it with `pip install xgboost`."
    ) from e


# =========================
# RNA structure / user utilities
# =========================
import RNA
import sylib


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"sklearn   = {sklearn_version}")
print(f"optuna    = {optuna.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"xgboost   = {xgb.__version__}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Progress bar & logging
# =========================
progress_bar = sylib.utils.ProgressBar()

logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)

logger = logging.getLogger(__name__)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python    = 3.11.15
pandas    = 2.3.3
numpy     = 2.4.6
scipy     = 1.17.1
sklearn   = 1.9.0
optuna    = 4.9.0
lightgbm  = 4.6.0
xgboost   = 3.2.0
ViennaRNA = 2.7.2
sylib     = 0.3.0.dev0+ae18bb2


In [3]:
# ============================================
# Benchmark settings
# ============================================

DATA_DIR = Path("../data/raw")

# prediction target
score_col_name = "PR_var"

# set to "gene_id" later if doing group-based split
group_col_name = None

# train/test split
test_data_ratio = 0.20

# reproducibility
random_state = 0

# CV
k_cv = 10

# parallelism
n_jobs = 10

# hyperparameter tuning
n_trials = 20

# species datasets and species-specific RNA folding temperature
species_config = {
    "AT21": {
        "file": DATA_DIR / "AT21.PR_var.dev_data.RNA.seq_data.tsv.gz",
        "rna_temp": 22,
    },
    "NB21": {
        "file": DATA_DIR / "NB21.PR_var.dev_data.RNA.seq_data.tsv.gz",
        "rna_temp": 22,
    },
    "OS21": {
        "file": DATA_DIR / "OS21.PR_var.dev_data.RNA.seq_data.tsv.gz",
        "rna_temp": 25,
    },
}

print("Species configuration:")
for species, cfg in species_config.items():
    print(f"{species}: file={cfg['file']}, rna_temp={cfg['rna_temp']}")

Species configuration:
AT21: file=../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz, rna_temp=22
NB21: file=../data/raw/NB21.PR_var.dev_data.RNA.seq_data.tsv.gz, rna_temp=22
OS21: file=../data/raw/OS21.PR_var.dev_data.RNA.seq_data.tsv.gz, rna_temp=25


In [4]:
# ============================================
# Check input files
# ============================================

for species, cfg in species_config.items():
    file_path = cfg["file"]
    print(species)
    print(" ", file_path)
    print(" exists:", file_path.exists())

AT21
  ../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
 exists: True
NB21
  ../data/raw/NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
 exists: True
OS21
  ../data/raw/OS21.PR_var.dev_data.RNA.seq_data.tsv.gz
 exists: True


In [5]:
# ============================================
# Basic sequence helpers
# ============================================

def normalize_rna(seq):
    if pd.isna(seq):
        return ""
    return str(seq).upper().replace("T", "U")


def cat_mrna(utr5, cds, utr3):
    return normalize_rna(utr5) + normalize_rna(cds) + normalize_rna(utr3)

In [6]:
# ============================================
# Model registry: default benchmark models
# ============================================

def get_default_models(random_state=0, n_jobs=10):
    return {
        "PLS": PLSRegression(n_components=5),

        "Ridge": Ridge(alpha=1.0),

        "Lasso": Lasso(
            alpha=0.001,
            max_iter=10000,
            random_state=random_state,
        ),

        "MLP": MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            alpha=0.0001,
            learning_rate_init=0.001,
            max_iter=500,
            random_state=random_state,
        ),

        "DecisionTree": DecisionTreeRegressor(
            random_state=random_state,
        ),

        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            n_jobs=n_jobs,
        ),

        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=1.0,
            colsample_bytree=1.0,
            random_state=random_state,
            n_jobs=n_jobs,
        ),

        "LightGBM": LGBMRegressor(
            objective="regression",
            n_estimators=300,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=n_jobs,
            verbosity=-1,
        ),
    }


default_models = get_default_models(
    random_state=random_state,
    n_jobs=n_jobs,
)

list(default_models.keys())

['PLS',
 'Ridge',
 'Lasso',
 'MLP',
 'DecisionTree',
 'RandomForest',
 'XGBoost',
 'LightGBM']

In [10]:
# ============================================
# Load one species
# ============================================

def load_species_data(species, species_config):

    file_path = species_config[species]["file"]

    seq_df, metadata = sylib.fileio.load_df(
        str(file_path)
    )

    metadata.print_minimum_data(
        label=species,
        logger=logger,
        logging_level="info",
    )

    return seq_df, metadata

# Check Data Structure
seq_df, metadata = load_species_data(
    "AT21",
    species_config
)

print(seq_df.shape)

display(seq_df.head())

2026-06-14 18:05:21,516     INFO: AT21 |   name = AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:05:21,517     INFO: AT21 |    md5 = 463f63c03bb16bfb05e1f6ea025947b8
2026-06-14 18:05:21,517     INFO: AT21 | md5_gz = 7940f4f1aa886ba02c249422c074027e


(65, 11)


,var_id,trans_id,gene_id,usage,PR_var,5'UTR,CDS,3'UTR,n_5'UTR_introns,n_CDS_introns,n_3'UTR_introns
0,AT1G07770.2.2409543.2408264,AT1G07770.2,AT1G07770,0.043245,0.408580,CUCUUCUUCCUCUAAUUGCUUUUCUCCGUCACUGAGUACCUUGCCU...,AUGGUAAGAAUCAGUGUUCUUAACGAUGCUCUCAAGAGCAUGUACA...,AGUUUUAUUUUAUGGGGAAACAGAUUUUCAUUGAGUUAUUUUUAAC...,1,2,0
1,AT1G07770.2.2409543.2408265,AT1G07770.2,AT1G07770,0.035668,0.447437,CUCUUCUUCCUCUAAUUGCUUUUCUCCGUCACUGAGUACCUUGCCU...,AUGGUAAGAAUCAGUGUUCUUAACGAUGCUCUCAAGAGCAUGUACA...,AGUUUUAUUUUAUGGGGAAACAGAUUUUCAUUGAGUUAUUUUUAAC...,1,2,0
2,AT1G07770.3.2409543.2408265,AT1G07770.3,AT1G07770,0.037044,0.626719,CUCUUCUUCCUCUAAUUGCUUUUCUCCGUCACUGAGUACCUUGCCU...,AUGGUAAGAAUCAGUGUUCUUAACGAUGCUCUCAAGAGCAUGUACA...,AGUUUUAUUUUAUGGGGAAACAGAUUUUCAUUGAGUUAUUUUUAAC...,1,2,0
3,AT1G07820.1.2421940.2421219,AT1G07820.1,AT1G07820,0.111209,0.522178,GUCAAAUUCAAUUGAUCCUCUCUCCAAAUCAUCUUAAAAGUAUCUU...,AUGUCGGGAAGAGGAAAGGGAGGAAAAGGAUUGGGAAAGGGAGGAG...,UCCGAUUUGGGGGAUUAGGGUUUAUGCAAGUUUGGGGAUUUCUUCU...,0,0,0
4,AT1G07820.2.2421937.2421219,AT1G07820.2,AT1G07820,0.079275,0.490502,AAAUUCAAUUGAUCCUCUCUCCAAAUCAUCUUAAAAUCACAGAUCU...,AUGUCGGGAAGAGGAAAGGGAGGAAAAGGAUUGGGAAAGGGAGGAG...,UCCGAUUUGGGGGAUUAGGGUUUAUGCAAGUUUGGGGAUUUCUUCU...,1,0,0


In [15]:
# ============================================
# Feature Evaluation: Definition of Features
# Only benchmark features are included.
# ============================================

def eval_length(seq_list, region_name, logger=logger):
    return pd.DataFrame({
        f"{region_name}.Length": [
            len(seq) if isinstance(seq, str) else 0
            for seq in seq_list
        ]
    })


def eval_kmer_freq(seq_list, region_name, is_rna_seq=True, k=1, logger=logger):
    count_dict = sylib.sequtils.count_nuc_k_mer_of_seq_list(
        seq_list,
        is_rna_seq=is_rna_seq,
        k=k,
    )

    feat_df = pd.DataFrame({
        f"{region_name}.{key}-freq": count_dict[key]
        for key in count_dict
    })

    seq_len_sr = pd.Series([
        len(seq) if isinstance(seq, str) else 0
        for seq in seq_list
    ])

    seq_len_sr[seq_len_sr == 0] = 1

    for col_name in feat_df.columns:
        feat_df[col_name] /= seq_len_sr
        feat_df[col_name] *= 1000

    return feat_df


def eval_s_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("G") + seq.count("C")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.S-freq": v})


def eval_y_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("T") + seq.count("U") + seq.count("C")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.Y-freq": v})


def eval_k_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("G") + seq.count("T") + seq.count("U")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.K-freq": v})


def eval_n_uaug(utr5_seq_list, region_name, is_rna_seq=True, logger=logger):
    if is_rna_seq:
        return pd.DataFrame({
            f"{region_name}.uAUG": [
                seq.count("AUG") if isinstance(seq, str) else 0
                for seq in utr5_seq_list
            ]
        })
    else:
        return pd.DataFrame({
            f"{region_name}.uATG": [
                seq.count("ATG") if isinstance(seq, str) else 0
                for seq in utr5_seq_list
            ]
        })


def eval_n_uorf(utr5_seq_list, region_name, is_rna_seq=True, logger=logger):
    feat_df_dict = {
        f"{region_name}.uORF-NO": [],
        f"{region_name}.uORF-IO": [],
        f"{region_name}.uORF-OO": [],
    }

    if is_rna_seq:
        start_codon = "AUG"
        stop_codons = {"UAA", "UGA", "UAG"}
    else:
        start_codon = "ATG"
        stop_codons = {"TAA", "TGA", "TAG"}

    for seq in utr5_seq_list:
        seq = seq if isinstance(seq, str) else ""
        L = len(seq)

        n_no = 0
        n_io = 0
        n_oo = 0

        for i in range(L - 2):
            if seq[i:i+3] == start_codon:
                has_stop = False

                for j in range(i + 3, L - 2, 3):
                    if seq[j:j+3] in stop_codons:
                        has_stop = True
                        break

                if has_stop:
                    n_no += 1
                elif (L - i) % 3 == 0:
                    n_io += 1
                else:
                    n_oo += 1

        feat_df_dict[f"{region_name}.uORF-NO"].append(n_no)
        feat_df_dict[f"{region_name}.uORF-IO"].append(n_io)
        feat_df_dict[f"{region_name}.uORF-OO"].append(n_oo)

    return pd.DataFrame(feat_df_dict)


def eval_n_daug(utr3_seq_list, region_name, is_rna_seq=True, logger=logger):
    if is_rna_seq:
        return pd.DataFrame({
            f"{region_name}.dAUG": [
                seq.count("AUG") if isinstance(seq, str) else 0
                for seq in utr3_seq_list
            ]
        })
    else:
        return pd.DataFrame({
            f"{region_name}.dATG": [
                seq.count("ATG") if isinstance(seq, str) else 0
                for seq in utr3_seq_list
            ]
        })


def eval_n_dorf(utr3_seq_list, region_name, is_rna_seq=True, logger=logger):
    feat_df_dict = {
        f"{region_name}.dORF": [],
        f"{region_name}.dORF-NS": [],
    }

    if is_rna_seq:
        start_codon = "AUG"
        stop_codons = {"UAA", "UGA", "UAG"}
    else:
        start_codon = "ATG"
        stop_codons = {"TAA", "TGA", "TAG"}

    for seq in utr3_seq_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq + "AA"
        L = len(seq)

        n_no = 0
        n_ns = 0

        for i in range(L - 2):
            if seq[i:i+3] == start_codon:
                has_stop = False

                for j in range(i + 3, L - 2, 3):
                    if seq[j:j+3] in stop_codons:
                        has_stop = True
                        break

                if has_stop:
                    n_no += 1
                else:
                    n_ns += 1

        feat_df_dict[f"{region_name}.dORF"].append(n_no)
        feat_df_dict[f"{region_name}.dORF-NS"].append(n_ns)

    return pd.DataFrame(feat_df_dict)

# ============================================
# Feature Evaluation: CDS coding features
# Only benchmark non-MFE features are included here.
# MFE is precomputed separately by script.
# ============================================

def eval_codon_usage(cds_seq_list, region_name, is_rna_seq=True, should_remove_stop_codons=False, logger=logger):
    mer3_list = sylib.sequtils.get_k_mer_nuc_list(3, is_rna_seq=is_rna_seq)
    aa_codon_dict = sylib.sequtils.get_aa_codon_dict(
        is_rna_seq=is_rna_seq,
        code_type="3-letter",
    )

    n_codons_dict = {mer3: [0] * len(cds_seq_list) for mer3 in mer3_list}

    for i, cds_seq in enumerate(cds_seq_list):
        cds_seq = cds_seq if isinstance(cds_seq, str) else ""
        for j in range(0, len(cds_seq), 3):
            codon = cds_seq[j:j+3]
            if codon in n_codons_dict:
                n_codons_dict[codon][i] += 1

    codons_df = pd.DataFrame(n_codons_dict)
    usage_df_dict = {}

    for aa, codon_list in aa_codon_dict.items():
        aa_sr = pd.Series([0] * len(cds_seq_list))

        for codon in codon_list:
            aa_sr += codons_df[codon]

        aa_sr[aa_sr == 0] = 1

        for codon in codon_list:
            key = f"{region_name}.{aa}-{codon}"
            usage_df_dict[key] = (codons_df[codon] / aa_sr) * 1000

    if is_rna_seq:
        usage_df_dict.pop(f"{region_name}.Met-AUG", None)
        usage_df_dict.pop(f"{region_name}.Trp-UGG", None)

        if should_remove_stop_codons:
            for c in ("UAA", "UGA", "UAG"):
                usage_df_dict.pop(f"{region_name}.Ter-{c}", None)
    else:
        usage_df_dict.pop(f"{region_name}.Met-ATG", None)
        usage_df_dict.pop(f"{region_name}.Trp-TGG", None)

        if should_remove_stop_codons:
            for c in ("TAA", "TGA", "TAG"):
                usage_df_dict.pop(f"{region_name}.Ter-{c}", None)

    return pd.DataFrame(usage_df_dict)


def wobble_freq(cds_list, region_label="CDS"):
    alphabet = ["A", "U", "G", "C"]
    cols = {f"{region_label}.wobble_pct_{b}": [] for b in alphabet}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        wobble_bases = [
            seq[i]
            for i in range(2, len(seq), 3)
            if i < len(seq)
        ]

        total = len(wobble_bases)

        for b in alphabet:
            cols[f"{region_label}.wobble_pct_{b}"].append(
                wobble_bases.count(b) / total if total > 0 else 0.0
            )

    return pd.DataFrame(cols)


CODON_TO_AA_RNA = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "X", "UAG": "X",
    "UGU": "C", "UGC": "C", "UGA": "X", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

AA_LIST = sorted(set(CODON_TO_AA_RNA.values()))


def aa_freq_from_cds(cds_list, region_label="CDS"):
    cols = {f"{region_label}.aa_pct_{aa}": [] for aa in AA_LIST}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        counts = {aa: 0 for aa in AA_LIST}
        total = 0

        for i in range(0, len(seq) - 2, 3):
            codon = seq[i:i+3]
            aa = CODON_TO_AA_RNA.get(codon)

            if aa is None:
                continue

            counts[aa] += 1
            total += 1

        for aa in AA_LIST:
            cols[f"{region_label}.aa_pct_{aa}"].append(
                counts[aa] / total if total > 0 else 0.0
            )

    return pd.DataFrame(cols)


DICODONS_DNA = [
    "AGGCGA", "AGGCGG", "ATACGA", "ATACGG", "CGAATA",
    "CGACCG", "CGACGA", "CGACGG", "CGACTG", "CGAGCG",
    "CTCATA", "CTCCCG", "CTGATA", "CTGCCG", "CTGCGA",
    "CTGCTG", "CTTCTG", "GTACCG", "GTACGA", "GTGCGA",
]

DICODONS_RNA = [d.replace("T", "U") for d in DICODONS_DNA]


def dicodon_counts_rna(cds_list, region_label="CDS", normalized=False):
    cols = {f"{region_label}.dicodon_{d}": [] for d in DICODONS_RNA}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        counts = {d: 0 for d in DICODONS_RNA}
        total = 0

        for i in range(0, len(seq) - 5, 3):
            dicodon = seq[i:i+6]

            if len(dicodon) == 6:
                total += 1

                if dicodon in counts:
                    counts[dicodon] += 1

        for d in DICODONS_RNA:
            value = counts[d] / total if (normalized and total > 0) else counts[d]
            cols[f"{region_label}.dicodon_{d}"].append(value)

    return pd.DataFrame(cols)

In [19]:
# ============================================
# Feature Evaluation: Execution of non-MFE features
# ============================================

def build_non_mfe_features(df):
    parts = []

    # 1. Region length
    parts += [
        eval_length(df["5'UTR"], "5'UTR"),
        eval_length(df["CDS"],   "CDS"),
        eval_length(df["3'UTR"], "3'UTR"),
    ]

    # 2. Region nucleotide / k-mer composition
    for k in [1, 2, 3]:
        parts += [
            eval_kmer_freq(df["5'UTR"], "5'UTR", is_rna_seq=True, k=k),
            eval_kmer_freq(df["CDS"],   "CDS",   is_rna_seq=True, k=k),
            eval_kmer_freq(df["3'UTR"], "3'UTR", is_rna_seq=True, k=k),
        ]

    # 3. Degenerate nucleotide class features
    parts += [
        eval_s_freq(df["5'UTR"], "5'UTR"),
        eval_y_freq(df["5'UTR"], "5'UTR"),
        eval_k_freq(df["5'UTR"], "5'UTR"),
        eval_s_freq(df["3'UTR"], "3'UTR"),
        eval_y_freq(df["3'UTR"], "3'UTR"),
        eval_k_freq(df["3'UTR"], "3'UTR"),
    ]

    # 4. Upstream/downstream ORF-related features
    parts += [
        eval_n_uaug(df["5'UTR"], "5'UTR", is_rna_seq=True),
        eval_n_uorf(df["5'UTR"], "5'UTR", is_rna_seq=True),
        eval_n_daug(df["3'UTR"], "3'UTR", is_rna_seq=True),
        eval_n_dorf(df["3'UTR"], "3'UTR", is_rna_seq=True),
    ]

    # 5. CDS coding features
    parts += [
        eval_codon_usage(df["CDS"], "CDS", is_rna_seq=True, should_remove_stop_codons=True),
        wobble_freq(df["CDS"], "CDS"),
        aa_freq_from_cds(df["CDS"], "CDS"),
        dicodon_counts_rna(df["CDS"], "CDS", normalized=True),
    ]

    x_feat_df = pd.concat(parts, axis=1)

    return x_feat_df

x_non_mfe_df = build_non_mfe_features(seq_df)

print(x_non_mfe_df.shape)
display(x_non_mfe_df.head())

# ============================================
# Check non-MFE feature matrix
# ============================================

print("shape:", x_non_mfe_df.shape)
print("missing values:", x_non_mfe_df.isna().sum().sum())
print("infinite values:", np.isinf(x_non_mfe_df.to_numpy()).sum())
print("duplicated columns:", x_non_mfe_df.columns.duplicated().sum())

display(x_non_mfe_df.describe().T.head(20))

(65, 372)


,5'UTR.Length,CDS.Length,3'UTR.Length,5'UTR.A-freq,5'UTR.U-freq,5'UTR.G-freq,5'UTR.C-freq,CDS.A-freq,CDS.U-freq,CDS.G-freq,...,CDS.dicodon_CUCAUA,CDS.dicodon_CUCCCG,CDS.dicodon_CUGAUA,CDS.dicodon_CUGCCG,CDS.dicodon_CUGCGA,CDS.dicodon_CUGCUG,CDS.dicodon_CUUCUG,CDS.dicodon_GUACCG,CDS.dicodon_GUACGA,CDS.dicodon_GUGCGA
0,86,393,149,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,86,393,148,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,89,393,148,202.247191,337.078652,179.775281,280.898876,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,195,312,216,251.282051,405.128205,143.589744,200.000000,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,73,312,216,356.164384,315.068493,136.986301,191.780822,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


shape: (65, 372)
missing values: 0
infinite values: 0
duplicated columns: 0


,count,mean,std,min,25%,50%,75%,max
5'UTR.Length,65.0,62.307692,34.017423,5.000000,44.000000,58.000000,78.000000,195.000000
CDS.Length,65.0,389.030769,202.720788,186.000000,312.000000,369.000000,432.000000,1425.000000
3'UTR.Length,65.0,185.830769,71.527217,11.000000,148.000000,166.000000,216.000000,414.000000
5'UTR.A-freq,65.0,295.096829,99.719133,125.000000,240.000000,281.250000,333.333333,600.000000
5'UTR.U-freq,65.0,277.747351,92.000735,0.000000,232.876712,300.000000,321.428571,440.677966
5'UTR.G-freq,65.0,192.255735,65.545943,57.142857,154.929577,186.440678,234.042553,428.571429
5'UTR.C-freq,65.0,234.900084,88.311725,0.000000,187.500000,225.352113,290.697674,416.666667
CDS.A-freq,65.0,289.661042,55.539720,179.487179,255.747126,279.898219,333.333333,386.206897
CDS.U-freq,65.0,225.058616,53.183031,96.774194,193.384224,219.966159,264.367816,314.553991
CDS.G-freq,65.0,268.307326,28.481474,208.791209,261.754386,270.114943,280.459770,327.586207


In [20]:
# ============================================
# MFE precompute wrapper
# ============================================

def get_mfe_file_path(seq_file):
    seq_file = Path(seq_file)
    return seq_file.with_name(
        seq_file.name.replace(".RNA.seq_data.tsv.gz", ".MFE.tsv.gz")
    )


def ensure_mfe_file(seq_file, temperature, n_workers=10, force=False):
    seq_file = Path(seq_file)
    mfe_file = get_mfe_file_path(seq_file)

    if mfe_file.exists() and not force:
        logger.info("MFE file already exists: %s", mfe_file)
        return mfe_file

    cmd = [
        sys.executable,
        "../scripts/precompute_mfe.py",
        str(seq_file),
        str(mfe_file),
        "--temperature",
        str(temperature),
        "-p",
        str(n_workers),
    ]

    logger.info("Running MFE script:")
    logger.info(" ".join(cmd))

    subprocess.run(cmd, check=True)

    return mfe_file

In [21]:
# ============================================
# Test MFE precompute wrapper: AT21
# ============================================

species = "AT21"

mfe_file = ensure_mfe_file(
    seq_file=species_config[species]["file"],
    temperature=species_config[species]["rna_temp"],
    n_workers=n_jobs,
    force=False,
)

print(mfe_file)
print(mfe_file.exists())

2026-06-14 18:38:00,368     INFO: Running MFE script:
2026-06-14 18:38:00,369     INFO: /home/ha-ibnu/miniconda3/envs/ibnu/bin/python ../scripts/precompute_mfe.py ../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz ../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz --temperature 22 -p 10
2026-06-14 18:38:01,947     INFO: loading input: ../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:38:01,954     INFO: Input data |   name = AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:38:01,954     INFO: Input data |    md5 = 463f63c03bb16bfb05e1f6ea025947b8
2026-06-14 18:38:01,954     INFO: Input data | md5_gz = 7940f4f1aa886ba02c249422c074027e
2026-06-14 18:38:01,955     INFO: calculating MFE.5'UTR
2026-06-14 18:38:02,007     INFO: completed 65 / 65
2026-06-14 18:38:02,014     INFO: calculating MFE.CDS
2026-06-14 18:38:03,241     INFO: completed 65 / 65
2026-06-14 18:38:03,248     INFO: calculating MFE.3'UTR
2026-06-14 18:38:03,442     INFO: completed 65 / 65
2026-06-14 18:38:0

../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz
True


2026-06-14 18:38:05,643     INFO: completed 65 / 65
2026-06-14 18:38:05,650     INFO: writing output: ../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz
2026-06-14 18:38:05,653     INFO: done


In [22]:
# ============================================
# Load and check MFE features: AT21
# ============================================

mfe_df, mfe_metadata = sylib.fileio.load_df(str(mfe_file))

mfe_metadata.print_minimum_data(
    label=f"{species} MFE",
    logger=logger,
    logging_level="info",
)

print(mfe_df.shape)
display(mfe_df.head())

2026-06-14 18:38:31,511     INFO: AT21 MFE |   name = AT21.PR_var.dev_data.MFE.tsv.gz
2026-06-14 18:38:31,511     INFO: AT21 MFE |    md5 = 919a6c2b813f7270ab43f352253f0f39
2026-06-14 18:38:31,512     INFO: AT21 MFE | md5_gz = 1127b9e8e7b833e9e6a4005fb68cf16d


(65, 7)


,var_id,trans_id,gene_id,MFE.5'UTR,MFE.CDS,MFE.3'UTR,MFE.mRNA
0,AT1G07770.2.2409543.2408264,AT1G07770.2,AT1G07770,-28.57,-160.449997,-39.34,-242.389999
1,AT1G07770.2.2409543.2408265,AT1G07770.2,AT1G07770,-28.57,-160.449997,-39.34,-242.389999
2,AT1G07770.3.2409543.2408265,AT1G07770.3,AT1G07770,-30.15,-160.449997,-39.34,-243.250000
3,AT1G07820.1.2421940.2421219,AT1G07820.1,AT1G07820,-45.48,-122.290001,-57.73,-265.089996
4,AT1G07820.2.2421937.2421219,AT1G07820.2,AT1G07820,-10.29,-122.290001,-57.73,-217.289993


In [23]:
# ============================================
# Merge non-MFE and MFE features
# ============================================

mfe_feature_cols = [
    "MFE.5'UTR",
    "MFE.CDS",
    "MFE.3'UTR",
    "MFE.mRNA",
]

# safety check: row order should match original data
assert (seq_df["var_id"].values == mfe_df["var_id"].values).all()

x_feat_df = pd.concat(
    [
        x_non_mfe_df.reset_index(drop=True),
        mfe_df[mfe_feature_cols].reset_index(drop=True),
    ],
    axis=1,
)

print(x_feat_df.shape)
print("missing values:", x_feat_df.isna().sum().sum())
print("infinite values:", np.isinf(x_feat_df.to_numpy()).sum())
print("duplicated columns:", x_feat_df.columns.duplicated().sum())

display(x_feat_df.head())

(65, 376)
missing values: 0
infinite values: 0
duplicated columns: 0


,5'UTR.Length,CDS.Length,3'UTR.Length,5'UTR.A-freq,5'UTR.U-freq,5'UTR.G-freq,5'UTR.C-freq,CDS.A-freq,CDS.U-freq,CDS.G-freq,...,CDS.dicodon_CUGCGA,CDS.dicodon_CUGCUG,CDS.dicodon_CUUCUG,CDS.dicodon_GUACCG,CDS.dicodon_GUACGA,CDS.dicodon_GUGCGA,MFE.5'UTR,MFE.CDS,MFE.3'UTR,MFE.mRNA
0,86,393,149,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,-28.57,-160.449997,-39.34,-242.389999
1,86,393,148,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,-28.57,-160.449997,-39.34,-242.389999
2,89,393,148,202.247191,337.078652,179.775281,280.898876,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,-30.15,-160.449997,-39.34,-243.250000
3,195,312,216,251.282051,405.128205,143.589744,200.000000,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,-45.48,-122.290001,-57.73,-265.089996
4,73,312,216,356.164384,315.068493,136.986301,191.780822,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,-10.29,-122.290001,-57.73,-217.289993


In [24]:
# ============================================
# Build complete feature matrix for one species
# ============================================

def build_species_feature_matrix(species, species_config, n_workers=10, force_mfe=False):
    logger.info("Building feature matrix for %s", species)

    seq_df, metadata = load_species_data(species, species_config)

    # non-MFE features
    x_non_mfe_df = build_non_mfe_features(seq_df)

    # MFE features
    mfe_file = ensure_mfe_file(
        seq_file=species_config[species]["file"],
        temperature=species_config[species]["rna_temp"],
        n_workers=n_workers,
        force=force_mfe,
    )

    mfe_df, mfe_metadata = sylib.fileio.load_df(str(mfe_file))

    # safety check
    assert (seq_df["var_id"].values == mfe_df["var_id"].values).all()

    mfe_feature_cols = [
        "MFE.5'UTR",
        "MFE.CDS",
        "MFE.3'UTR",
        "MFE.mRNA",
    ]

    x_feat_df = pd.concat(
        [
            x_non_mfe_df.reset_index(drop=True),
            mfe_df[mfe_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )

    # quality checks
    n_missing = x_feat_df.isna().sum().sum()
    n_infinite = np.isinf(x_feat_df.to_numpy()).sum()
    n_duplicated = x_feat_df.columns.duplicated().sum()

    logger.info(
        "%s features: shape=%s, missing=%s, infinite=%s, duplicated_cols=%s",
        species,
        x_feat_df.shape,
        n_missing,
        n_infinite,
        n_duplicated,
    )

    if n_missing > 0:
        raise ValueError(f"{species}: missing values found in feature matrix")
    if n_infinite > 0:
        raise ValueError(f"{species}: infinite values found in feature matrix")
    if n_duplicated > 0:
        raise ValueError(f"{species}: duplicated feature columns found")

    y_array = np.array(seq_df[score_col_name])

    return seq_df, x_feat_df, y_array

In [25]:
# ============================================
# Test feature matrix generation for all species
# ============================================

species_feature_data = {}

for species in species_config:
    seq_df_sp, x_feat_df_sp, y_array_sp = build_species_feature_matrix(
        species,
        species_config,
        n_workers=n_jobs,
        force_mfe=False,
    )

    species_feature_data[species] = {
        "seq_df": seq_df_sp,
        "x_feat_df": x_feat_df_sp,
        "y_array": y_array_sp,
    }

    print(species, seq_df_sp.shape, x_feat_df_sp.shape, y_array_sp.shape)

2026-06-14 18:45:21,731     INFO: Building feature matrix for AT21
2026-06-14 18:45:21,741     INFO: AT21 |   name = AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:21,742     INFO: AT21 |    md5 = 463f63c03bb16bfb05e1f6ea025947b8
2026-06-14 18:45:21,743     INFO: AT21 | md5_gz = 7940f4f1aa886ba02c249422c074027e
2026-06-14 18:45:21,826     INFO: MFE file already exists: ../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz
2026-06-14 18:45:21,829     INFO: AT21 features: shape=(65, 376), missing=0, infinite=0, duplicated_cols=0
2026-06-14 18:45:21,829     INFO: Building feature matrix for NB21
2026-06-14 18:45:21,851     INFO: NB21 |   name = NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:21,852     INFO: NB21 |    md5 = cb81365e736f8802756f1a587f31c2db
2026-06-14 18:45:21,852     INFO: NB21 | md5_gz = 15627c4927492c9bddc2ef2a107d1986
2026-06-14 18:45:21,997     INFO: Running MFE script:
2026-06-14 18:45:21,998     INFO: /home/ha-ibnu/miniconda3/envs/ibnu/bin/python ../scrip

AT21 (65, 11) (65, 376) (65,)


2026-06-14 18:45:23,628     INFO: loading input: ../data/raw/NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:23,646     INFO: Input data |   name = NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:23,646     INFO: Input data |    md5 = cb81365e736f8802756f1a587f31c2db
2026-06-14 18:45:23,646     INFO: Input data | md5_gz = 15627c4927492c9bddc2ef2a107d1986
2026-06-14 18:45:23,647     INFO: calculating MFE.5'UTR
2026-06-14 18:45:23,718     INFO: completed 100 / 500
2026-06-14 18:45:23,741     INFO: completed 200 / 500
2026-06-14 18:45:23,767     INFO: completed 300 / 500
2026-06-14 18:45:23,789     INFO: completed 400 / 500
2026-06-14 18:45:23,867     INFO: completed 500 / 500
2026-06-14 18:45:23,874     INFO: calculating MFE.CDS
2026-06-14 18:45:25,479     INFO: completed 100 / 500
2026-06-14 18:45:27,035     INFO: completed 200 / 500
2026-06-14 18:45:28,863     INFO: completed 300 / 500
2026-06-14 18:45:30,210     INFO: completed 400 / 500
2026-06-14 18:45:31,922    

NB21 (500, 11) (500, 376) (500,)


2026-06-14 18:45:52,447     INFO: loading input: ../data/raw/OS21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:52,456     INFO: Input data |   name = OS21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:52,456     INFO: Input data |    md5 = 8d0fecf64cd0d7e31a3e43be59292070
2026-06-14 18:45:52,456     INFO: Input data | md5_gz = 7c64f7c0dd96839119b21df2563d9eec
2026-06-14 18:45:52,456     INFO: calculating MFE.5'UTR
2026-06-14 18:45:52,532     INFO: completed 100 / 117
2026-06-14 18:45:52,549     INFO: completed 117 / 117
2026-06-14 18:45:52,558     INFO: calculating MFE.CDS
2026-06-14 18:45:54,956     INFO: completed 100 / 117
2026-06-14 18:45:55,419     INFO: completed 117 / 117
2026-06-14 18:45:55,428     INFO: calculating MFE.3'UTR
2026-06-14 18:45:55,779     INFO: completed 100 / 117
2026-06-14 18:45:55,819     INFO: completed 117 / 117
2026-06-14 18:45:55,828     INFO: calculating MFE.mRNA
2026-06-14 18:46:01,193     INFO: completed 100 / 117
2026-06-14 18:46:01,976 

OS21 (117, 11) (117, 376) (117,)


In [26]:
# ============================================
# Feature matrix summary across species
# ============================================

summary_rows = []

for species, data in species_feature_data.items():
    x_feat_df = data["x_feat_df"]

    summary_rows.append({
        "species": species,
        "n_samples": x_feat_df.shape[0],
        "n_features": x_feat_df.shape[1],
        "n_missing": x_feat_df.isna().sum().sum(),
        "n_infinite": np.isinf(x_feat_df.to_numpy()).sum(),
        "n_duplicated_cols": x_feat_df.columns.duplicated().sum(),
        "n_constant_features": (x_feat_df.nunique() <= 1).sum(),
    })

feature_summary_df = pd.DataFrame(summary_rows)

display(feature_summary_df)

,species,n_samples,n_features,n_missing,n_infinite,n_duplicated_cols,n_constant_features
0,AT21,65,376,0,0,0,14
1,NB21,500,376,0,0,0,2
2,OS21,117,376,0,0,0,13


In [28]:
# ============================================
# Preprocessing: split, outlier removal, transform
# ============================================

xform_method = "yeo-johnson"
outlier_removal_method = "Z-score"
z_score_expected_n = 0.5
lof_contamination = "auto"

outlier_det_col_list = [
    "5'UTR.Length",
    "CDS.Length",
    "3'UTR.Length",
]

def detect_outliers(
    xformed_train_array,
    xformed_test_array,
    removal_method,
    z_score_expected_n=0.5,
    lof_contamination="auto",
):
    if removal_method is None:
        train_bool_array = np.ones(len(xformed_train_array), dtype=bool)
        test_bool_array = np.ones(len(xformed_test_array), dtype=bool)

    elif removal_method == "Z-score":
        min_thresh = stats.norm.ppf(
            (z_score_expected_n / 2) / len(xformed_train_array),
            loc=0,
            scale=1,
        )
        max_thresh = -min_thresh

        train_bool_array = (
            (xformed_train_array >= min_thresh)
            & (xformed_train_array <= max_thresh)
        )
        test_bool_array = (
            (xformed_test_array >= min_thresh)
            & (xformed_test_array <= max_thresh)
        )

    elif removal_method == "LOF":
        clf = LocalOutlierFactor(
            novelty=True,
            contamination=lof_contamination,
        ).fit(xformed_train_array.reshape(-1, 1))

        train_bool_array = clf.predict(xformed_train_array.reshape(-1, 1)) == 1
        test_bool_array = clf.predict(xformed_test_array.reshape(-1, 1)) == 1

    else:
        raise ValueError(f"Unknown outlier removal method: {removal_method}")

    return train_bool_array, test_bool_array